# Self-play benchmark - parallel vs vectorized (Option B)

**Group 501** | Colman College | DL Final Project

Decides whether to flip a 9x9 run to `self_play_mode: "vectorized"`.

Run the cells top to bottom. Section 1 mirrors the training notebooks: it
installs `requirements.txt` into this kernel (the container's `/usr/bin/python3`
ships without torch) and then reports what torch will actually run on.

**Requires an idle GPU** - section 2 checks. Run this next to a live training job
and both engines are measured under contention, which invalidates the result.

---
## 1. Environment Setup
Run once per kernel session.

In [ ]:
# 1.1 - Locate repo and install dependencies

import os, sys

# Find repo root (works from any starting directory)
REPO_DIR = None
for candidate in [
    os.getcwd(),                               # already in repo root
    os.path.join(os.getcwd(), "dl-quoridor"),  # Jupyter workspace root
    os.path.dirname(os.getcwd()),              # running from notebooks/
]:
    if os.path.exists(os.path.join(candidate, ".git")):
        REPO_DIR = candidate
        break

assert REPO_DIR is not None, (
    "Could not find dl-quoridor repo. "
    "Make sure the notebook is inside the repo or one level above it."
)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo: {REPO_DIR}")
print(f"Python: {sys.executable}")

# Deliberately NOT running `git checkout . && git pull` like the training
# notebooks do: this benchmark is meant to be run from a feature branch, and
# `git checkout .` would discard your working changes. Confirm the branch:
!git -C {REPO_DIR} log --oneline -1
!git -C {REPO_DIR} status -sb | head -1

# Install requirements (--ignore-installed handles system-managed packages)
!pip install -r requirements.txt -q --ignore-installed
print("Dependencies installed.")

In [ ]:
# 1.2 - Detect hardware; confirm torch sees the GPU

from scripts.bench_self_play import describe_device, run_bench

DEVICE = describe_device("auto")

If 1.2 printed `cuda_available=False`, **stop here**. Both engines would be
CPU-bound, the vectorized driver would run unpipelined, and the numbers would
say nothing about how this box behaves during real training.

---
## 2. Is the GPU free?

Any python process listed below means something is already training. Benchmarking
under contention penalises the worker-pool path most, which biases the result
toward vectorized.

In [ ]:
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

---
## 3. Smoke test (~1 min)

Tiny run, just to confirm both engines execute in this kernel. The ratio here is
meaningless - 5x5, 25 sims, no warmup convergence.

In [ ]:
_ = run_bench(games=4, sims=25, num_workers=4, max_moves=40, warmup_games=1)

---
## 4. The measurement

One rule: **keep width, scale depth.**

- **Width** (`games=50`, `vec_games=64`, `num_workers=32`) is what separates the
  two engines. Both cap concurrency at the game count - `min(vec_games, games)`
  and `min(num_workers, games)` - so lowering `games` benchmarks a narrower
  engine than you intend to run. `run_bench` warns if you do.
- **Depth** (`sims`, `max_moves`) is paid identically by both engines, so
  reducing it scales both sides down and preserves the ratio. It costs you
  absolute evals/s, which you don't need for this decision.

So: production width, reduced depth. Expect **10-20 minutes**.

Progress prints on every completed game with an ETA, plus a liveness line every
30s - if output stops entirely, it really is stuck.

In [ ]:
PROXY = dict(
    board_size=9, num_players=4, walls=10,
    num_channels=128, num_res_blocks=8,
    vec_games=64, num_workers=32, batch_size=256,
    explore_moves=15, device="auto",
    games=50,                      # full width - do not lower
    sims=200, max_moves=120,       # reduced depth - both engines equally
    repeat=1,
    warmup_games=0,                # at these durations CUDA start-up is noise
)

r = run_bench(**PROXY)
ratio = r["parallel"] / r["vectorized"]
print(f"\nparallel/vectorized = {ratio:.2f}x  "
      f"({'vectorized' if ratio > 1 else 'parallel'} faster)")

### Reading the result

- **ratio > 1.3** - vectorized clearly wins. Switch.
- **ratio < 0.77** - parallel clearly wins. Do not switch; the vectorized driver
  is single-process, so all 64 games' tree-walking runs on one core, and at 9x9
  N=4 move generation (a BFS per player for wall legality) can outweigh the
  batching win. This is the CPU-bound outcome the PR flagged.
- **in between** - too close to call at this depth. Re-run the cell below with
  the ordering swapped; if the winner flips, the difference is noise and you
  should stay on parallel, since it is the incumbent.

In [ ]:
r2 = run_bench(**dict(PROXY, order="vectorized,parallel"))
ratio2 = r2["parallel"] / r2["vectorized"]
print(f"\nparallel first   : {ratio:.2f}x")
print(f"vectorized first : {ratio2:.2f}x")
print("AGREE" if (ratio > 1) == (ratio2 > 1) else "FLIPS WITH ORDER - treat as no difference")

---
## 5. Full depth (optional - only for writeup numbers)

Production `sims=800` / `max_moves=300`. This gives real evals/s figures rather
than a ratio, and takes **1-2 hours**. The decision in section 4 does not need it.

In [ ]:
FULL = dict(PROXY, sims=800, max_moves=300)
r_full = run_bench(**FULL)
print(f"\nparallel/vectorized = {r_full['parallel'] / r_full['vectorized']:.2f}x")

---
## 6. If you interrupt a run

A hard-killed benchmark leaves worker processes holding CUDA state, which wedges
the GPU for the next run - `run_notebook.sh`'s preflight exists for this. Always:

```
!pkill -f ipykernel
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv
```

then Kernel → Restart before running again.

## Switching the run

Only if section 4 says vectorized wins:

1. Stop the run.
2. `configs/config_9x9.json` → `"self_play_mode": "vectorized"`, `"vec_games": 64`.
3. Resume - weights persist via `latest.pt`; only the replay buffer refills.

The games played after the switch differ from what parallel would have produced
(different exploration-noise stream), but their quality does not: the vectorized
driver runs exact sequential MCTS with no virtual-loss approximation.